# Día 5 · Unidad 5 — POO aplicada (I): clases, atributos y métodos

**Objetivos de aprendizaje**
- Explicar qué es la programación orientada a objetos y cuándo aporta frente a un diccionario o una lista.
- Definir clases con atributos de instancia y atributos de clase, y saber cuándo usar cada uno.
- Escribir métodos de instancia.
- Escribir métodos de clase (`@classmethod`), incluyendo un patrón de fábrica con contador automático.
- Implementar métodos dunder básicos (`__repr__`, `__eq__`, `__lt__`) para que los objetos se comporten de forma predecible.

**Duración estimada:** 75-90 minutos.

**Requisitos previos:** Día 4 (funciones, `*args`/`**kwargs`, docstrings, typing).

## 1. Qué es la programación orientada a objetos

La POO es un paradigma para estructurar código agrupando datos y comportamiento relacionados en **objetos**. Un objeto podría representar una póliza: tiene propiedades (asegurado, prima, coberturas) y puede ejecutar acciones (renovarse, anularse, cambiar de cobertura).

Hasta ahora, para representar una póliza habrías usado un diccionario:

In [ ]:
poliza_1 = {
    "asegurado": "Anna Puig",
    "prima": 480.0,
    "coberturas": ["RC", "DAÑOS"],
}

poliza_2 = {
    "asegurado": "Marc Soler",
    "prima_mensual": 40.0,  # ojo: aquí la clave se llama distinto
    "coberturas": ["RC"],
}

print(poliza_1["prima"])
print(poliza_2.get("prima"))  # None: la clave no existe en poliza_2, y no salta ningún error

Este segundo `print` es el problema: `poliza_2` no tiene una prima "de verdad", tiene una `"prima_mensual"`. Nada te avisa del error hasta que, mucho más adelante en el programa, alguien intenta sumar primas y una de ellas es `None`.

Con un diccionario, la estructura de una póliza vive solo en la cabeza de quien escribió el código. Con una **clase**, la estructura queda fijada en un solo sitio: todas las pólizas tendrán los mismos atributos, se llamen igual y sean del tipo que sean.

> Dicho de otra manera: una clase es como una plantilla o formulario con campos fijos. Una instancia es un formulario ya rellenado con datos concretos.

### Clase vs. instancia

- Una **clase** especifica qué atributos y qué comportamiento tendrán todos los objetos de ese tipo — por ejemplo, "toda `Poliza` tiene un asegurado, una prima y unas coberturas".
- Una **instancia** es un objeto concreto creado a partir de una clase, con datos reales — por ejemplo, la póliza concreta de Anna Puig.

### Cómo definir una clase

Se usa la palabra clave `class`, seguida del nombre (en `CapitalizedWords` por convención) y dos puntos:

In [ ]:
class Poliza:
    pass

Esta clase todavía no hace nada interesante. Le falta decir qué atributos tiene toda `Poliza`.

## 2. Atributos de instancia: `__init__`

Los atributos que debe tener **cada instancia** se definen en el método `__init__`. Python lo llama automáticamente cada vez que creas un objeto nuevo.

El primer parámetro de `__init__` (y de cualquier método de instancia) siempre es `self`: es una referencia al objeto que se está creando o usando en ese momento.

In [ ]:
class Poliza:
    def __init__(self, asegurado: str, prima: float, coberturas: list[str]):
        self.asegurado = asegurado
        self.prima = prima
        self.coberturas = coberturas

Cada línea `self.algo = algo` crea un atributo de instancia. `asegurado`, `prima` y `coberturas` variarán entre pólizas: son propios de cada instancia.

## 3. Instanciar un objeto

In [ ]:
poliza_1 = Poliza(asegurado="Anna Puig", prima=480.0, coberturas=["RC", "DAÑOS"])
poliza_2 = Poliza(asegurado="Marc Soler", prima=320.0, coberturas=["RC"])

print(poliza_1.asegurado, poliza_1.prima)
print(poliza_2.asegurado, poliza_2.prima)

`poliza_1` y `poliza_2` son objetos independientes: cambiar `poliza_1.prima` no afecta a `poliza_2.prima`. Cada instancia vive en su propia dirección de memoria, aunque ambas comparten la misma clase.

## 4. Atributos de clase

Un atributo de clase tiene el **mismo valor para todas las instancias**, salvo que alguna lo sobreescriba explícitamente. Se define dentro del cuerpo de la clase, fuera de `__init__`.

In [ ]:
class Poliza:

    aseguradora = "CAC Assicurances"  # atributo de clase: igual para todas las pólizas

    def __init__(self, asegurado: str, prima: float, coberturas: list[str]):
        self.asegurado = asegurado
        self.prima = prima
        self.coberturas = coberturas


poliza_1 = Poliza(asegurado="Anna Puig", prima=480.0, coberturas=["RC", "DAÑOS"])
poliza_2 = Poliza(asegurado="Marc Soler", prima=320.0, coberturas=["RC"])

print(poliza_1.aseguradora, poliza_2.aseguradora)

> 📊 En Excel esto sería como una celda "aseguradora" fija en la cabecera de la hoja, común a todas las filas de pólizas, frente a una columna "prima" que cambia fila a fila.

> ⚠️ **Error típico**: usar un atributo de clase **mutable** (una lista o un diccionario) pensando que cada instancia tendrá "la suya". No es así: si no se define dentro de `__init__`, todas las instancias comparten el mismo objeto en memoria.

In [ ]:
class PolizaConBug:

    coberturas = []  # ¡mal! esto es UN solo objeto lista, compartido por todas las instancias

    def __init__(self, asegurado: str):
        self.asegurado = asegurado

    def anadir_cobertura(self, cobertura: str) -> None:
        self.coberturas.append(cobertura)  # modifica el atributo de CLASE, no uno propio


poliza_a = PolizaConBug(asegurado="Anna Puig")
poliza_b = PolizaConBug(asegurado="Marc Soler")

poliza_a.anadir_cobertura("RC")

print("Coberturas de Marc:", poliza_b.coberturas)  # "RC" aparece aquí también, aunque nunca se lo añadimos

`poliza_b` nunca recibió una cobertura, y aun así `"RC"` aparece en su lista: ambas instancias comparten la misma lista en memoria porque `coberturas` se definió como atributo de clase. La corrección es simple: mover la inicialización a `__init__`, donde cada instancia crea su propia lista nueva.

In [ ]:
class Poliza:

    aseguradora = "CAC Assicurances"

    def __init__(self, asegurado: str, prima: float, coberturas: list[str] | None = None):
        self.asegurado = asegurado
        self.prima = prima
        self.coberturas = coberturas if coberturas is not None else []  # lista propia de cada instancia


poliza_a = Poliza(asegurado="Anna Puig", prima=480.0)
poliza_b = Poliza(asegurado="Marc Soler", prima=320.0)

poliza_a.coberturas.append("RC")

print("Coberturas de Anna:", poliza_a.coberturas)
print("Coberturas de Marc:", poliza_b.coberturas)

**Para ti:** crea una clase `Asegurado` con atributos de instancia `nombre` y `dni`, y un atributo de clase `pais = "ES"` común a todos. Instancia dos asegurados distintos y comprueba que `pais` es el mismo para ambos pero `nombre`/`dni` no.

In [ ]:
# Tu código aquí

## 5. Métodos de instancia

Son funciones definidas dentro de la clase que actúan sobre una instancia concreta. Como `__init__`, su primer parámetro siempre es `self`.

In [ ]:
class Poliza:

    aseguradora = "CAC Assicurances"

    def __init__(self, asegurado: str, prima: float, vigente: bool = True):
        self.asegurado = asegurado
        self.prima = prima
        self.vigente = vigente

    def anular(self) -> None:
        """Marca la póliza como no vigente."""
        self.vigente = False

    def renovar(self, incremento_pct: float = 0.0) -> None:
        """Renueva la póliza, aplicando opcionalmente un incremento porcentual a la prima."""
        self.vigente = True
        self.prima = round(self.prima * (1 + incremento_pct / 100), 2)


poliza_1 = Poliza(asegurado="Anna Puig", prima=480.0)
poliza_1.anular()
print("Vigente tras anular:", poliza_1.vigente)

poliza_1.renovar(incremento_pct=5)
print("Vigente tras renovar:", poliza_1.vigente, "- Nueva prima:", poliza_1.prima)

## 6. Métodos de clase: `@classmethod`

Un método de clase recibe `cls` (la propia clase) en lugar de `self` (una instancia). Se usa sobre todo para **fábricas**: formas alternativas de construir objetos.

Supongamos que cada póliza necesita un número identificativo único, autoincremental, sin que el usuario tenga que inventárselo a mano.

In [ ]:
class Poliza:

    _siguiente_numero = 1  # atributo de clase: contador compartido

    def __init__(self, asegurado: str, numero_poliza: int):
        self.asegurado = asegurado
        self.numero_poliza = numero_poliza

    @classmethod
    def nueva(cls, asegurado: str) -> "Poliza":
        """Crea una póliza asignando automáticamente el siguiente número disponible."""
        poliza = cls(asegurado=asegurado, numero_poliza=cls._siguiente_numero)
        cls._siguiente_numero += 1
        return poliza


poliza_1 = Poliza.nueva(asegurado="Anna Puig")
poliza_2 = Poliza.nueva(asegurado="Marc Soler")

print(poliza_1.numero_poliza, poliza_2.numero_poliza)

> 📊 En Excel esto sería como una macro que genera automáticamente el siguiente número de fila/póliza cada vez que das de alta un registro, sin que tengas que escribirlo tú y sin riesgo de repetir un número.

Si en cambio dejamos que sea quien llama al constructor el que decida el número, se abre la puerta a duplicados — es responsabilidad de quien usa la clase acertar, y eso es justo lo que un método de fábrica evita.

**Para ti:** añade a la clase `Poliza` un método de clase `desde_diccionario(cls, datos: dict)` que reciba un diccionario con claves `"asegurado"` y `"numero_poliza"` y devuelva una instancia. Pruébalo con `Poliza.desde_diccionario({"asegurado": "Laura Ros", "numero_poliza": 99})`.

In [ ]:
# Tu código aquí

## 7. Métodos dunder básicos

Los métodos "dunder" (*double underscore*, `__algo__`) son los que Python llama internamente cuando usas operadores o funciones integradas (`print`, `==`, `<`, `len`...) sobre tus objetos. Definirlos te permite decidir cómo se comportan tus clases ante esas operaciones.

In [ ]:
class Poliza:

    def __init__(self, asegurado: str, numero_poliza: int, prima: float):
        self.asegurado = asegurado
        self.numero_poliza = numero_poliza
        self.prima = prima

    def __repr__(self) -> str:
        return f"Poliza(numero={self.numero_poliza}, asegurado={self.asegurado!r}, prima={self.prima})"

    def __eq__(self, otra: object) -> bool:
        if not isinstance(otra, Poliza):
            return NotImplemented
        return self.numero_poliza == otra.numero_poliza

    def __lt__(self, otra: "Poliza") -> bool:
        return self.prima < otra.prima


poliza_1 = Poliza(asegurado="Anna Puig", numero_poliza=1, prima=480.0)
poliza_2 = Poliza(asegurado="Anna Puig (duplicado en el sistema)", numero_poliza=1, prima=480.0)
poliza_3 = Poliza(asegurado="Marc Soler", numero_poliza=2, prima=320.0)

print(poliza_1)                    # usa __repr__
print(poliza_1 == poliza_2)        # True: mismo numero_poliza, aunque son objetos distintos
print(poliza_3 < poliza_1)         # True: 320.0 < 480.0
print(sorted([poliza_1, poliza_3], key=lambda p: p.prima))

Sin `__repr__`, `print(poliza_1)` mostraría algo como `<__main__.Poliza object at 0x...>`, una dirección de memoria inútil para depurar. Sin `__eq__`, `poliza_1 == poliza_2` compararía identidad de objeto (`is`), no si representan la misma póliza — y sería `False` aunque ambas tengan el mismo `numero_poliza`. Volveremos sobre esta diferencia entre `==` y `is` en el bloque de revisión crítica del siguiente notebook.

**Para ti:** añade a `Poliza` un método `__len__` que devuelva el número de coberturas contratadas (necesitarás un atributo `coberturas: list[str]` en `__init__`). Comprueba que `len(mi_poliza)` funciona.

In [ ]:
# Tu código aquí

## Resumen

- Una clase es una plantilla; una instancia es un objeto concreto creado a partir de ella, con `__init__` fijando sus atributos propios.
- Los **atributos de instancia** (definidos con `self.algo = ...` dentro de `__init__`) son propios de cada objeto; los **atributos de clase** (definidos en el cuerpo de la clase) son compartidos por todas las instancias.
- Un atributo de clase **mutable** (lista, diccionario) es un error típico: todas las instancias comparten el mismo objeto salvo que se inicialice dentro de `__init__`.
- Los **métodos de instancia** reciben `self` y actúan sobre un objeto concreto; los **métodos de clase** (`@classmethod`) reciben `cls` y son útiles como fábricas alternativas al constructor por defecto.
- Los **métodos dunder** (`__repr__`, `__eq__`, `__lt__`, `__len__`...) controlan cómo se comportan tus objetos ante `print`, `==`, `<`, `len()`, `sorted()`, etc.
- `__eq__` bien definido compara **valores** (por ejemplo, el mismo `numero_poliza`); sin él, Python compara identidad de objeto, que casi nunca es lo que quieres para modelar datos de negocio.

**Siguiente:** `02_teoria_properties_herencia.ipynb`, donde veremos `@property`/`@setter`, herencia entre clases de pólizas, y el bloque de revisión crítica de código con fallo sutil.